In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Two-Layer Hierarchical Disaster Triage Pipeline (`models/train_disaster_triage_3class_exp.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and trains a **Two-Layer Hierarchical Cascade Model** across 8 core arrival triage features:

```mermaid
flowchart TD
    Patient["Incoming Emergency Patient (8 Vitals & Demographics)"] --> L1["Layer 1: LightGBM (ESI 1 Screening)"]
    L1 -- "P(ESI 1) >= 0.50" --> ESI1["🚨 ESI 1 (Immediate Resuscitation / RED)"]
    L1 -- "P(ESI 1) < 0.50" --> L2["Layer 2: Random Forest (Acuity Differentiation)"]
    L2 -- "P(ESI 2-3) >= 0.50" --> ESI23["⚠️ ESI 2-3 (Emergent & Urgent / YELLOW)"]
    L2 -- "P(ESI 2-3) < 0.50" --> ESI45["🟢 ESI 4-5 (Semi-urgent & Non-urgent / GREEN)"]
```

### 🎯 Tiered Target Formulations
1. **Layer 1 (L1 - Resuscitation Screening)**: `LightGBM (Random Undersampling)`
   - `Class 1: ESI 1 (Immediate Resuscitation)` ($5,271$ visits, $\sim 0.94\%$)
   - `Class 0: NOT ESI 1 (ESI 2–5)` ($552,758$ visits, $\sim 99.06\%$)
2. **Layer 2 (L2 - Sub-Acuity Differentiation)**: `Random Forest (Balanced Weights)`
   - `Class 0: ESI 2-3 (Emergent / Urgent)` ($440,059$ visits, $\sim 79.61\%$ of L2)
   - `Class 1: ESI 4-5 (Semi-urgent / Non-urgent)` ($112,699$ visits, $\sim 20.39\%$ of L2)

### 🩺 8 Arrival Triage Features
1. `age`
2. `cc_breathingdifficulty`
3. `gender` (0 = Female, 1 = Male)
4. `triage_vital_hr` (Heart Rate)
5. `triage_vital_sbp` (Systolic Blood Pressure)
6. `triage_vital_dbp` (Diastolic Blood Pressure)
7. `triage_vital_rr` (Respiratory Rate)
8. `triage_vital_o2` (Oxygen Saturation - SpO2)

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns (all NAs preserved for SimpleImputer)\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve from R, Partition & Median Imputation
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Define 3-Tier Categorical Acuity Labels
# 0: ESI 1 (RED), 1: ESI 2-3 (YELLOW), 2: ESI 4-5 (GREEN)
y_3tier_all = np.zeros(len(esi_all), dtype=np.int32)
y_3tier_all[esi_all == 1] = 0                          # ESI 1
y_3tier_all[np.isin(esi_all, [2, 3])] = 1              # ESI 2-3
y_3tier_all[np.isin(esi_all, [4, 5])] = 2              # ESI 4-5
TIER_LABELS = ['ESI 1 (RED)', 'ESI 2-3 (YELLOW)', 'ESI 4-5 (GREEN)']

print("=========================================================================")
print("               5v_cleandf 3-TIER DISASTER TRIAGE COHORT")
print("=========================================================================")
print(f"Total Valid ESI Visits: {len(esi_all):,}")
print(f"  * Tier 0 [ESI 1 (RED)]    : {np.sum(y_3tier_all == 0):,} ({np.mean(y_3tier_all == 0)*100:.2f}%)")
print(f"  * Tier 1 [ESI 2-3 (YELLOW)]: {np.sum(y_3tier_all == 1):,} ({np.mean(y_3tier_all == 1)*100:.2f}%)")
print(f"  * Tier 2 [ESI 4-5 (GREEN)] : {np.sum(y_3tier_all == 2):,} ({np.mean(y_3tier_all == 2)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(esi_all)), test_size=0.30, stratify=y_3tier_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_3tier_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

esi_tr  = esi_all[itr]
esi_val = esi_all[iva]
esi_te  = esi_all[ite]

y_3tier_tr  = y_3tier_all[itr]
y_3tier_val = y_3tier_all[iva]
y_3tier_te  = y_3tier_all[ite]

# Fit SimpleImputer strictly on Training partition
print("Fitting SimpleImputer(strategy='median') on Training set...")
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(raw_tr)
X_val   = imputer.transform(raw_val)
X_test  = imputer.transform(raw_te)

print(f"✓ Partition Shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Train Layer 1 (LightGBM with RUS: ESI 1 vs NOT ESI 1)
# ---------------------------------------------------------------------------
# Target L1: 1 = ESI 1, 0 = NOT ESI 1 (ESI 2-5)
y_l1_tr  = np.where(esi_tr == 1, 1, 0)
y_l1_val = np.where(esi_val == 1, 1, 0)
y_l1_te  = np.where(esi_te == 1, 1, 0)

print("Applying RandomUnderSampler on Layer 1 Training Split (ESI 1 vs NOT ESI 1)...")
rus = RandomUnderSampler(random_state=42)
X_l1_tr_rus, y_l1_tr_rus = rus.fit_resample(X_train, y_l1_tr)
print(f"✓ L1 Undersampled Training Shape: {X_l1_tr_rus.shape} ({np.sum(y_l1_tr_rus == 1):,} ESI 1 vs {np.sum(y_l1_tr_rus == 0):,} NOT ESI 1)")

print("\nTraining Layer 1 LightGBM Binary Classifier...")
l1_model = LGBMClassifier(
    objective='binary',
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)
l1_model.fit(
    X_l1_tr_rus,
    y_l1_tr_rus,
    eval_set=[(X_val, y_l1_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=True)]
)

p_l1_te = l1_model.predict_proba(X_test)[:, 1]
pred_l1_te = l1_model.predict(X_test)
rec_l1 = recall_score(y_l1_te == 1, pred_l1_te == 1, zero_division=0)
spec_l1 = recall_score(y_l1_te == 0, pred_l1_te == 0, zero_division=0)
print(f"✓ Layer 1 (LightGBM) Holdout Metrics: ESI 1 Recall={rec_l1*100:.2f}%, Specificity={spec_l1*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Train Layer 2 (Random Forest: ESI 2-3 vs ESI 4-5 on Sub-Acuity Visits)
# ---------------------------------------------------------------------------
# Filter training partition strictly to sub-acuity visits (ESI 2, 3, 4, 5)
l2_mask_tr  = np.isin(esi_tr, [2, 3, 4, 5])
l2_mask_val = np.isin(esi_val, [2, 3, 4, 5])
l2_mask_te  = np.isin(esi_te, [2, 3, 4, 5])

X_l2_tr  = X_train[l2_mask_tr]
X_l2_val = X_val[l2_mask_val]
X_l2_te  = X_test[l2_mask_te]

# Target L2: 0 = ESI 2-3 (Emergent/Urgent / YELLOW), 1 = ESI 4-5 (Semi-urgent/Non-urgent / GREEN)
y_l2_tr  = np.where(np.isin(esi_tr[l2_mask_tr], [4, 5]), 1, 0)
y_l2_val = np.where(np.isin(esi_val[l2_mask_val], [4, 5]), 1, 0)
y_l2_te  = np.where(np.isin(esi_te[l2_mask_te], [4, 5]), 1, 0)

print(f"Layer 2 Dataset Breakdown (ESI 2-3 vs ESI 4-5):")
print(f"  * Training Set  : {len(y_l2_tr):,} visits ({np.sum(y_l2_tr == 0):,} ESI 2-3 vs {np.sum(y_l2_tr == 1):,} ESI 4-5)")
print(f"  * Validation Set: {len(y_l2_val):,} visits ({np.sum(y_l2_val == 0):,} ESI 2-3 vs {np.sum(y_l2_val == 1):,} ESI 4-5)")
print(f"  * Holdout Test  : {len(y_l2_te):,} visits ({np.sum(y_l2_te == 0):,} ESI 2-3 vs {np.sum(y_l2_te == 1):,} ESI 4-5)")

print("\nTraining Layer 2 Random Forest Classifier (ESI 2-3 vs ESI 4-5)...")
l2_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
l2_model.fit(X_l2_tr, y_l2_tr)

p_l2_te = l2_model.predict_proba(X_l2_te)[:, 1]
pred_l2_te = l2_model.predict(X_l2_te)
rec_esi45 = recall_score(y_l2_te == 1, pred_l2_te == 1, zero_division=0)
rec_esi23 = recall_score(y_l2_te == 0, pred_l2_te == 0, zero_division=0)
auc_l2    = roc_auc_score(y_l2_te, p_l2_te)

print(f"✓ Layer 2 (Random Forest) Holdout Metrics:")
print(f"  * ESI 2-3 (YELLOW) Recall : {rec_esi23*100:.2f}%")
print(f"  * ESI 4-5 (GREEN) Recall  : {rec_esi45*100:.2f}%")
print(f"  * Layer 2 ROC-AUC         : {auc_l2:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: End-to-End Cascaded 3-Tier Evaluation on Holdout Test Set
# ---------------------------------------------------------------------------
# Hierarchical Inference Function:
# 1. If Layer 1 predicts ESI 1 (p >= 0.50) -> Tier 0: ESI 1 (RED)
# 2. Else -> Layer 2 predicts (p >= 0.50 -> Tier 2: ESI 4-5, else -> Tier 1: ESI 2-3)
def predict_hierarchical(X_mat, l1, l2, tau_l1=0.50, tau_l2=0.50):
    p_l1 = l1.predict_proba(X_mat)[:, 1]
    is_esi1 = p_l1 >= tau_l1
    
    p_l2_green = l2.predict_proba(X_mat)[:, 1]
    is_green = p_l2_green >= tau_l2
    
    final_pred = np.where(is_esi1, 0, np.where(is_green, 2, 1))
    return final_pred, p_l1, p_l2_green

pred_3tier_te, p_l1_all_te, p_l2_all_te = predict_hierarchical(X_test, l1_model, l2_model)

# 3-Class Metrics Calculation
acc_3tier     = accuracy_score(y_3tier_te, pred_3tier_te)
bal_acc_3tier = balanced_accuracy_score(y_3tier_te, pred_3tier_te)
rec_per_class = recall_score(y_3tier_te, pred_3tier_te, average=None)
prec_per_class= precision_score(y_3tier_te, pred_3tier_te, average=None)
f1_per_class  = f1_score(y_3tier_te, pred_3tier_te, average=None)
macro_f1      = f1_score(y_3tier_te, pred_3tier_te, average='macro')
weighted_f1   = f1_score(y_3tier_te, pred_3tier_te, average='weighted')

report_rows = [
    {
        'Triage_Tier': 'ESI 1 (RED)',
        'True_Visits': int(np.sum(y_3tier_te == 0)),
        'Predicted_Visits': int(np.sum(pred_3tier_te == 0)),
        'Sensitivity (Recall)': round(rec_per_class[0], 4),
        'Precision': round(prec_per_class[0], 4),
        'F1_Score': round(f1_per_class[0], 4)
    },
    {
        'Triage_Tier': 'ESI 2-3 (YELLOW)',
        'True_Visits': int(np.sum(y_3tier_te == 1)),
        'Predicted_Visits': int(np.sum(pred_3tier_te == 1)),
        'Sensitivity (Recall)': round(rec_per_class[1], 4),
        'Precision': round(prec_per_class[1], 4),
        'F1_Score': round(f1_per_class[1], 4)
    },
    {
        'Triage_Tier': 'ESI 4-5 (GREEN)',
        'True_Visits': int(np.sum(y_3tier_te == 2)),
        'Predicted_Visits': int(np.sum(pred_3tier_te == 2)),
        'Sensitivity (Recall)': round(rec_per_class[2], 4),
        'Precision': round(prec_per_class[2], 4),
        'F1_Score': round(f1_per_class[2], 4)
    }
]

report_df = pd.DataFrame(report_rows)
print("=========================================================================================================")
print("    END-TO-END HIERARCHICAL 3-TIER EVALUATION (L1: LightGBM -> L2: Random Forest)")
print("=========================================================================================================")
print(f"Overall Accuracy         : {acc_3tier*100:.2f}%")
print(f"Macro Balanced Accuracy  : {bal_acc_3tier*100:.2f}%")
print(f"Macro F1-Score           : {macro_f1:.4f}")
print(f"Weighted F1-Score        : {weighted_f1:.4f}\n")
print(report_df.to_string(index=False))
print("=========================================================================================================\n")

print("Detailed Classification Report:")
print(classification_report(y_3tier_te, pred_3tier_te, target_names=TIER_LABELS, digits=4))

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'hierarchical_disaster_triage_3tier_report.csv')
report_df.to_csv(report_file, index=False)
print(f"3-tier metrics report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 3-Class Hierarchical Confusion Matrix Heatmap
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm_3tier = confusion_matrix(y_3tier_te, pred_3tier_te, labels=[0, 1, 2])
cm_3tier_norm = cm_3tier.astype('float') / cm_3tier.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8.5, 7))
annot = np.empty_like(cm_3tier, dtype=object)
for i in range(3):
    for j in range(3):
        annot[i, j] = f"{cm_3tier[i, j]:,}\n({cm_3tier_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_3tier_norm,
    annot=annot,
    fmt='',
    cmap='Blues',
    cbar=True,
    ax=ax,
    vmin=0,
    vmax=1,
    xticklabels=TIER_LABELS,
    yticklabels=TIER_LABELS
)

ax.set_title(
    f'Two-Layer Hierarchical Triage: 3-Tier Confusion Matrix\n'
    f'Accuracy: {acc_3tier*100:.2f}% | Macro Bal Acc: {bal_acc_3tier*100:.2f}% | Macro F1: {macro_f1:.4f}',
    fontsize=12,
    fontweight='bold',
    pad=12
)
ax.set_xlabel('Hierarchically Predicted Triage Category', fontsize=11, fontweight='bold')
ax.set_ylabel('True Emergency Acuity Category', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'disaster_triage_3class_hierarchical_confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'disaster_triage_3class_hierarchical_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'disaster_triage_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"3-Class Confusion Matrix saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: 2D Clinical Decision Boundary Visualizations for Both Layers (Test Set)
# ---------------------------------------------------------------------------
# Compute holdout test set median baseline across all 8 features
medians_test = np.median(X_test, axis=0)

# 4 Key Clinical Vital Planes for Decision Boundary Analysis
planes = [
    ('triage_vital_hr', 'triage_vital_sbp', 'Heart Rate (bpm)', 'Systolic BP (mmHg)', (30, 200), (50, 240), 'Hemodynamic Shock Plane (HR vs SBP)'),
    ('triage_vital_rr', 'triage_vital_o2', 'Respiratory Rate (bpm)', 'SpO2 (%)', (6, 45), (70, 100), 'Hypoxia & Respiratory Failure Plane (RR vs SpO2)'),
    ('triage_vital_hr', 'triage_vital_rr', 'Heart Rate (bpm)', 'Respiratory Rate (bpm)', (30, 200), (6, 45), 'Cardiopulmonary Distress Plane (HR vs RR)'),
    ('age', 'triage_vital_sbp', 'Age (years)', 'Systolic BP (mmHg)', (18, 95), (50, 240), 'Vascular Tone Across Age Plane (Age vs SBP)')
]

# Figure 1: Layer 1 (LightGBM ESI 1 Decision Boundaries on Test Set)
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

np.random.seed(42)
test_idx_esi1 = np.where(y_3tier_te == 0)[0]
test_idx_other = np.where(y_3tier_te != 0)[0]
test_idx_other_sample = np.random.choice(test_idx_other, min(1500, len(test_idx_other)), replace=False)

for idx, (f1_name, f2_name, x_lbl, y_lbl, x_lim, y_lim, plane_title) in enumerate(planes):
    f1_idx = FEATURES.index(f1_name)
    f2_idx = FEATURES.index(f2_name)
    
    xx, yy = np.meshgrid(np.linspace(x_lim[0], x_lim[1], 150), np.linspace(y_lim[0], y_lim[1], 150))
    grid_points = np.tile(medians_test, (xx.size, 1))
    grid_points[:, f1_idx] = xx.ravel()
    grid_points[:, f2_idx] = yy.ravel()
    
    zz = l1_model.predict_proba(grid_points)[:, 1].reshape(xx.shape)
    
    ax = axes[idx]
    cf = ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), cmap='RdYlBu_r', alpha=0.65, vmin=0, vmax=1)
    cs = ax.contour(xx, yy, zz, levels=[0.50], colors='black', linewidths=2.5, linestyles='--')
    ax.clabel(cs, inline=True, fontsize=10, fmt='P(ESI 1) = %.2f')
    
    ax.scatter(X_test[test_idx_other_sample, f1_idx], X_test[test_idx_other_sample, f2_idx], c='#1f77b4', alpha=0.35, s=14, label='NOT ESI 1 (Sample)')
    ax.scatter(X_test[test_idx_esi1, f1_idx], X_test[test_idx_esi1, f2_idx], c='#d62728', alpha=0.85, s=32, edgecolors='black', linewidth=0.5, label='ESI 1 (Resus)')
    
    ax.set_title(f'Layer 1 (LightGBM): {plane_title}', fontsize=11.5, fontweight='bold', pad=10)
    ax.set_xlabel(x_lbl, fontsize=10.5, fontweight='bold')
    ax.set_ylabel(y_lbl, fontsize=10.5, fontweight='bold')
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.grid(True, linestyle=':', alpha=0.4)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)
    cbar = plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Predicted P(ESI 1)', fontsize=9.5, fontweight='bold')

plt.suptitle('Layer 1 (LightGBM): 2D Decision Boundaries (ESI 1 Screening)', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
db_l1_path = os.path.join(plots_dir, 'esi1_lightgbm_decision_boundaries.png')
plt.savefig(db_l1_path, dpi=300, bbox_inches='tight')
plt.show()

# Figure 2: Layer 2 (Random Forest ESI 2-3 vs ESI 4-5 Decision Boundaries)
fig2, axes2 = plt.subplots(2, 2, figsize=(16, 14))
axes2 = axes2.flatten()

test_idx_esi23 = np.where(y_3tier_te == 1)[0]
test_idx_esi45 = np.where(y_3tier_te == 2)[0]
test_idx_esi23_sample = np.random.choice(test_idx_esi23, min(1200, len(test_idx_esi23)), replace=False)
test_idx_esi45_sample = np.random.choice(test_idx_esi45, min(1200, len(test_idx_esi45)), replace=False)

for idx, (f1_name, f2_name, x_lbl, y_lbl, x_lim, y_lim, plane_title) in enumerate(planes):
    f1_idx = FEATURES.index(f1_name)
    f2_idx = FEATURES.index(f2_name)
    
    xx, yy = np.meshgrid(np.linspace(x_lim[0], x_lim[1], 150), np.linspace(y_lim[0], y_lim[1], 150))
    grid_points = np.tile(medians_test, (xx.size, 1))
    grid_points[:, f1_idx] = xx.ravel()
    grid_points[:, f2_idx] = yy.ravel()
    
    zz_l2 = l2_model.predict_proba(grid_points)[:, 1].reshape(xx.shape)
    
    ax = axes2[idx]
    cf = ax.contourf(xx, yy, zz_l2, levels=np.linspace(0, 1, 11), cmap='YlGnBu', alpha=0.65, vmin=0, vmax=1)
    cs = ax.contour(xx, yy, zz_l2, levels=[0.50], colors='darkgreen', linewidths=2.5, linestyles='--')
    ax.clabel(cs, inline=True, fontsize=10, fmt='P(ESI 4-5) = %.2f')
    
    ax.scatter(X_test[test_idx_esi23_sample, f1_idx], X_test[test_idx_esi23_sample, f2_idx], c='#ff7f0e', alpha=0.45, s=16, label='ESI 2-3 (YELLOW)')
    ax.scatter(X_test[test_idx_esi45_sample, f1_idx], X_test[test_idx_esi45_sample, f2_idx], c='#2ca02c', alpha=0.55, s=16, label='ESI 4-5 (GREEN)')
    
    ax.set_title(f'Layer 2 (Random Forest): {plane_title}', fontsize=11.5, fontweight='bold', pad=10)
    ax.set_xlabel(x_lbl, fontsize=10.5, fontweight='bold')
    ax.set_ylabel(y_lbl, fontsize=10.5, fontweight='bold')
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.grid(True, linestyle=':', alpha=0.4)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9.5, frameon=True, framealpha=0.9)
    cbar = plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Predicted P(ESI 4-5)', fontsize=9.5, fontweight='bold')

plt.suptitle('Layer 2 (Random Forest): 2D Decision Boundaries (ESI 2-3 vs ESI 4-5)', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
db_l2_path = os.path.join(plots_dir, 'esi23_vs_esi45_rf_decision_boundaries.png')
plt.savefig(db_l2_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'esi23_vs_esi45_rf_decision_boundaries.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Layer 1 boundaries saved to: {db_l1_path}")
print(f"Layer 2 boundaries saved to: {db_l2_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Comparative Feature Importances (Layer 1 LightGBM vs Layer 2 RF)
# ---------------------------------------------------------------------------
l1_imp = l1_model.feature_importances_ / l1_model.feature_importances_.sum()
l2_imp = l2_model.feature_importances_ / l2_model.feature_importances_.sum()

imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Layer 1: LightGBM (ESI 1 Screening)': l1_imp,
    'Layer 2: Random Forest (ESI 2-3 vs 4-5)': l2_imp
})

imp_melted = pd.melt(imp_df, id_vars=['Feature'], var_name='Layer', value_name='Importance')

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=imp_melted, x='Feature', y='Importance', hue='Layer', palette=['#1f77b4', '#2ca02c'], ax=ax)
ax.set_title('Hierarchical Feature Importances: Layer 1 (LightGBM) vs Layer 2 (Random Forest)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Arrival Triage Feature', fontsize=11, fontweight='bold')
ax.set_ylabel('Normalized Importance', fontsize=11, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(loc='upper right', fontsize=10.5)

plt.tight_layout()
imp_file = os.path.join(plots_dir, 'hierarchical_layers_feature_importance.png')
plt.savefig(imp_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"Feature importance comparison saved to: {imp_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: Export Hierarchical Production Bundle & Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

hierarchical_bundle = {
    'l1_model': l1_model,
    'l2_model': l2_model,
    'imputer': imputer,
    'features': FEATURES,
    'tier_labels': TIER_LABELS,
    'predict_fn': predict_hierarchical
}

bundle_file = os.path.join(deploy_dir, 'disaster_triage_hierarchical_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(hierarchical_bundle, f)

manifest = dict(
    pipeline_architecture='Two_Layer_Hierarchical_Disaster_Triage',
    layer_1=dict(
        model='LGBMClassifier',
        objective='Binary_ESI1_Screening',
        sampling='RandomUnderSampler',
        best_iteration=int(l1_model.best_iteration_)
    ),
    layer_2=dict(
        model='RandomForestClassifier',
        objective='Sub_Acuity_ESI23_vs_ESI45',
        class_weight='balanced',
        n_estimators=200
    ),
    dataset='5v_cleandf_RData',
    total_visits=len(esi_all),
    tier_labels=TIER_LABELS,
    feature_order=FEATURES,
    n_features=len(FEATURES),
    hierarchical_test_metrics=report_rows,
    overall_accuracy=round(acc_3tier, 4),
    macro_balanced_accuracy=round(bal_acc_3tier, 4),
    macro_f1=round(macro_f1, 4)
)

manifest_file = os.path.join(deploy_dir, 'disaster_triage_hierarchical_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported Hierarchical Bundle  : {bundle_file}")
print(f"✓ Exported Hierarchical Manifest: {manifest_file}")